In [1]:
!pip install pandas datasets pyarrow 

!pip install transformers

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer

In [3]:
tokenizer = AutoTokenizer.from_pretrained("unsloth/DeepSeek-R1-Distill-Llama-8B")
max_length = 2048

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [6]:
dataset = load_dataset("parquet", data_files={"train": "/kaggle/input/datasets/rehanfargose/sc-dataset-1990-2025/SC_Parquet_Dataset_1990-2025.parquet"}, split="train")
dataset = dataset.filter(lambda x: x['full_text'] is not None and len(str(x['full_text'])) > 200)

Filter:   0%|          | 0/26518 [00:00<?, ? examples/s]

In [7]:
def tokenize_function(examples):
    return tokenizer(examples["full_text"], truncation=False)

In [8]:
tokenized_dataset = dataset.map(
    tokenize_function, 
    batched=True, 
    batch_size=50, 
    num_proc=4, 
    remove_columns=dataset.column_names,
    desc="Tokenizing legal texts"
)

Tokenizing legal texts (num_proc=4):   0%|          | 0/26515 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (237528 > 131072). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (221011 > 131072). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (144005 > 131072). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (164225 > 131072). Running this sequence through the model will result in indexing errors


In [9]:
# 3. Pack into 2048 blocks (DAPT style)
def group_texts(examples):
    # Concatenate all texts in the batch
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated[list(examples.keys())[0]])
    # Round down to nearest multiple of max_length
    total_length = (total_length // max_length) * max_length
    # Split into chunks
    result = {
        k: [t[i : i + max_length] for i in range(0, total_length, max_length)]
        for k, t in concatenated.items()
    }
    return result

In [10]:
packed_dataset = tokenized_dataset.map(
    group_texts, 
    batched=True, 
    batch_size=50, 
    num_proc=4,
    desc="Packing into 2048 chunks"
)

Packing into 2048 chunks (num_proc=4):   0%|          | 0/26515 [00:00<?, ? examples/s]

In [11]:
# 4. Save to Kaggle Output
packed_dataset.save_to_disk("pre_tokenized_dataset")
print(f"Success! Created {len(packed_dataset)} training sequences.")

Saving the dataset (0/3 shards):   0%|          | 0/112865 [00:00<?, ? examples/s]

Success! Created 112865 training sequences.
